In [1]:
import os
# Enforce single GPU execution to prevent multi-GPU conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import login, hf_hub_download
from sklearn.metrics import f1_score, precision_recall_fscore_support

# ==============================================================================
# GLOBAL DEBUG CONTROL
# Set to True for rapid pipeline verification, False for full training.
# ==============================================================================
DEBUG = False

DEBUG_CONFIG = {
    "train_samples": 64,
    "dev_samples": 32,
    "test_samples": 16,
    "teacher_epochs": 1,
    "student_epochs": 1,
    "batch_size": 4,
    "log_interval": 2
}

FULL_CONFIG = {
    "train_samples": None,  # Full dataset
    "dev_samples": None,
    "test_samples": None,
    "teacher_epochs": 3,
    "student_epochs": 5,
    "batch_size": 4,
    "log_interval": 20
}

CFG = DEBUG_CONFIG if DEBUG else FULL_CONFIG

print(f"=== Execution Mode: {'DEBUG (Fast Verification Run)' if DEBUG else 'FULL PRODUCTION RUN'} ===")
print(f"Configuration: {CFG}\n")

=== Execution Mode: FULL PRODUCTION RUN ===
Configuration: {'train_samples': None, 'dev_samples': None, 'test_samples': None, 'teacher_epochs': 3, 'student_epochs': 5, 'batch_size': 4, 'log_interval': 20}



In [ ]:
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO_TEACHER = "A-240242/legal-modern-decoder-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_lsi_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_5_LSI/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 5 LSI Dataset...")
train_data = load_lsi_file("Train/train.json")
dev_data = load_lsi_file("Train/dev.json")

print(f"Loaded {len(train_data)} raw training records and {len(dev_data)} raw dev records.")

# Extract unique target statute IDs
all_statutes_set = set()
for record in train_data + dev_data:
    all_statutes_set.update(record.get("labels", []))

ALL_STATUTE_IDS = sorted(list(all_statutes_set))
NUM_STATUTES = len(ALL_STATUTE_IDS)
STATUTE2IDX = {sid: idx for idx, sid in enumerate(ALL_STATUTE_IDS)}
IDX2STATUTE = {idx: sid for sid, idx in STATUTE2IDX.items()}

print(f"Total Unique Statute Classes Identified: {NUM_STATUTES}")

# Slice dataset if running in debug mode
if DEBUG and CFG["train_samples"]:
    train_data = train_data[:CFG["train_samples"]]
    dev_data = dev_data[:CFG["dev_samples"]]
    print(f"[DEBUG] Subsampled train dataset to {len(train_data)} and dev dataset to {len(dev_data)} records.")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Loaded 42750 raw training records and 10181 raw dev records.
Total Unique Statute Classes Identified: 100


In [3]:
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class LSIPyTorchDataset(Dataset):
    """
    PyTorch Dataset for Phase 5 Multi-Label Legal Statute Identification.
    Extracts head (facts) and tail (ruling/statutes) chunks for long legal documents.
    """
    def __init__(self, raw_records, tokenizer, statute2idx, num_statutes, max_length=1024, is_test=False):
        self.samples = []
        self.is_test = is_test
        
        for record in raw_records:
            case_id = record.get("id")
            doc_text = record.get("text", "")
            
            tokenized = tokenizer(doc_text, truncation=False, return_attention_mask=False)
            full_ids = tokenized["input_ids"]
            
            # Head/tail chunking strategy for long document handling
            if len(full_ids) <= max_length:
                chunks = [full_ids]
            else:
                first_chunk = full_ids[:max_length]
                last_chunk = full_ids[-max_length:]
                chunks = [first_chunk, last_chunk]
                
            label_vec = [0.0] * num_statutes
            if not is_test:
                for sid in record.get("labels", []):
                    if sid in statute2idx:
                        label_vec[statute2idx[sid]] = 1.0
                        
            for chunk in chunks:
                self.samples.append({
                    "id": case_id,
                    "input_ids": chunk,
                    "labels": label_vec
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def lsi_collate_fn(batch):
    input_ids_list = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    labels_list = [torch.tensor(item["labels"], dtype=torch.float32) for item in batch]
    case_ids = [item["id"] for item in batch]
    
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    
    attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()
    labels_tensor = torch.stack(labels_list)
    
    return {
        "id": case_ids,
        "input_ids": padded_input_ids,
        "attention_mask": attention_mask,
        "labels": labels_tensor
    }

print("Building DataLoaders...")
train_dataset = LSIPyTorchDataset(train_data, tokenizer, STATUTE2IDX, NUM_STATUTES)
dev_dataset = LSIPyTorchDataset(dev_data, tokenizer, STATUTE2IDX, NUM_STATUTES)

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, collate_fn=lsi_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=CFG["batch_size"] * 2, shuffle=False, collate_fn=lsi_collate_fn)

print(f"Created {len(train_dataset)} training chunk samples and {len(dev_dataset)} dev chunk samples.")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Building DataLoaders...


Created 59542 training chunk samples and 14725 dev chunk samples.


In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int):
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    cos = cos[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    sin = sin[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class ModernDecoderAttention(nn.Module):
    """Multi-Head Self Attention with Rotary Embeddings."""
    def __init__(self, hidden_size=768, num_heads=12):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        B, S, _ = x.shape
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        if cos is not None and sin is not None:
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
            
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            attn_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0
            scores = scores + attn_mask
            
        probs = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.hidden_size)
        
        return self.o_proj(attn_out)


class SwiGLUMLP(nn.Module):
    """SwiGLU Feed-Forward Layer."""
    def __init__(self, hidden_size=768, intermediate_size=2048):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class ModernDecoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = RMSNorm(hidden_size)
        self.self_attn = ModernDecoderAttention(hidden_size, num_heads)
        self.post_attention_layernorm = RMSNorm(hidden_size)
        self.mlp = SwiGLUMLP(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        residual = x
        x = self.input_layernorm(x)
        x = self.self_attn(x, attention_mask=attention_mask, cos=cos, sin=sin)
        x = residual + x
        
        residual = x
        x = self.post_attention_layernorm(x)
        x = self.mlp(x)
        x = residual + x
        return x


class ModernDecoderBackbone(nn.Module):
    """Modern Decoder Transformer Backbone."""
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            ModernDecoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(hidden_size)
        self.rotary_emb = RotaryEmbedding(hidden_size // num_heads)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        B, S = input_ids.shape
        x = self.embed_tokens(input_ids)
        cos, sin = self.rotary_emb(x, S)
        
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask, cos=cos, sin=sin)
            
        x = self.norm(x)
        return x


class CustomModernDecoderSequenceClassification(nn.Module):
    """Modern Decoder Multi-Label Classifier."""
    def __init__(self, vocab_size, num_labels, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.model = ModernDecoderBackbone(vocab_size, hidden_size, num_layers, num_heads, intermediate_size)
        self.score = nn.Linear(hidden_size, num_labels, bias=False)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, labels: torch.Tensor = None):
        hidden_states = self.model(input_ids, attention_mask=attention_mask)
        
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()
            sum_hidden = (hidden_states * mask_expanded).sum(dim=1)
            sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
            pooled = sum_hidden / sum_mask
        else:
            pooled = hidden_states.mean(dim=1)
            
        logits = self.score(pooled)
        
        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels.float())
            
        return {"loss": loss, "logits": logits} if loss is not None else logits

print("Custom Modern Decoder Architecture written from scratch successfully.")

Custom Modern Decoder Architecture written from scratch successfully.


In [5]:
class LormaAdapter(nn.Module):
    """Low-Rank Matrix Adaptation Layer."""
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.scaling * self.B(self.A(x))


class LoRMAWrappedLinear(nn.Module):
    """Wraps a base Linear layer with a LoRMA adapter."""
    def __init__(self, base_linear: nn.Linear, rank=16, alpha=32):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
            
        self.adapter = LormaAdapter(base_linear.in_features, base_linear.out_features, rank=rank, alpha=alpha)

    def forward(self, x):
        return self.base(x) + self.adapter(x)


# 1. Instantiate Teacher Architecture (Full Modern Decoder Size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher_model = CustomModernDecoderSequenceClassification(
    vocab_size=len(tokenizer), 
    num_labels=NUM_STATUTES, 
    hidden_size=768, 
    num_layers=6, 
    num_heads=12, 
    intermediate_size=2048
)

# 2. Fetch and Load Pre-trained Phase 1 Checkpoint Weights
print("Fetching Teacher Model Checkpoint Weights from Hub...")
model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO_TEACHER, filename="model.safetensors", token=HF_TOKEN)
state_dict = load_file(model_weight_path)
teacher_model.load_state_dict(state_dict, strict=False)
print("Loaded Phase 1 weights successfully into Teacher Model.")

# 3. Inject LoRMA Adapters into Teacher Model
print("Injecting LoRMA Adapters into Teacher Model...")
for param in teacher_model.parameters():
    param.requires_grad = False

# Unfreeze classification head
for param in teacher_model.score.parameters():
    param.requires_grad = True

# Wrap target projections in Teacher
for layer in teacher_model.model.layers:
    for target_attr in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        base_linear = getattr(layer.self_attn, target_attr)
        setattr(layer.self_attn, target_attr, LoRMAWrappedLinear(base_linear, rank=16, alpha=32))
        
    for target_attr in ["gate_proj", "up_proj", "down_proj"]:
        base_linear = getattr(layer.mlp, target_attr)
        setattr(layer.mlp, target_attr, LoRMAWrappedLinear(base_linear, rank=16, alpha=32))

for name, param in teacher_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

teacher_model.to(device)

teacher_total = sum(p.numel() for p in teacher_model.parameters())
teacher_trainable = sum(p.numel() for p in teacher_model.parameters() if p.requires_grad)

print(f"\n[Teacher Model Summary]:")
print(f"  - Total Parameters: {teacher_total:,}")
print(f"  - Trainable Parameters (LoRMA + Head): {teacher_trainable:,}")

Fetching Teacher Model Checkpoint Weights from Hub...


model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded Phase 1 weights successfully into Teacher Model.
Injecting LoRMA Adapters into Teacher Model...



[Teacher Model Summary]:
  - Total Parameters: 68,530,944
  - Trainable Parameters (LoRMA + Head): 1,477,632


In [6]:
CHECKPOINT_DIR = "./checkpoints_phase5"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
TEACHER_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_teacher_lorma.pt")

teacher_optimizer = torch.optim.AdamW(
    [p for p in teacher_model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01
)

best_teacher_f1 = 0.0
print(f"\nCommencing Teacher Model Fine-Tuning ({CFG['teacher_epochs']} Epochs)...")

for epoch in range(CFG["teacher_epochs"]):
    teacher_model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        teacher_optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs["loss"]
        
        loss.backward()
        teacher_optimizer.step()
        
        total_train_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Teacher | Epoch {epoch+1}/{CFG['teacher_epochs']} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    # Dev Set Validation
    teacher_model.eval()
    dev_logits, dev_targets = [], []
    with torch.no_grad():
        for batch in dev_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            outputs = teacher_model(input_ids=ids, attention_mask=mask)
            dev_logits.append(outputs["logits"].cpu() if isinstance(outputs, dict) else outputs.cpu())
            dev_targets.append(batch["labels"])
            
    dev_logits = torch.cat(dev_logits, dim=0).numpy()
    dev_targets = torch.cat(dev_targets, dim=0).numpy()
    probs = 1.0 / (1.0 + np.exp(-dev_logits))
    preds = (probs >= 0.5).astype(int)
    
    epoch_f1 = f1_score(dev_targets, preds, average="macro", zero_division=0)
    print(f"Teacher Epoch {epoch+1} Completed | Dev Macro-F1 (thresh=0.5): {epoch_f1:.4f}")
    
    if epoch_f1 > best_teacher_f1 or epoch == 0:
        best_teacher_f1 = epoch_f1
        trainable_state = {k: v.cpu().clone() for k, v in teacher_model.state_dict().items() if v.requires_grad}
        torch.save({"epoch": epoch + 1, "best_f1": best_teacher_f1, "state_dict": trainable_state}, TEACHER_CHECKPOINT_PATH)
        print(f"  [+] Saved Best Teacher Checkpoint to '{TEACHER_CHECKPOINT_PATH}'")


Commencing Teacher Model Fine-Tuning (3 Epochs)...


Teacher | Epoch 1/3 | Step 20/14886 | Loss: 0.1267


Teacher | Epoch 1/3 | Step 40/14886 | Loss: 0.1493


Teacher | Epoch 1/3 | Step 60/14886 | Loss: 0.1408


Teacher | Epoch 1/3 | Step 80/14886 | Loss: 0.1297


Teacher | Epoch 1/3 | Step 100/14886 | Loss: 0.1048


... (2,229 outputs left out)

Teacher | Epoch 3/3 | Step 14860/14886 | Loss: 0.2026


Teacher | Epoch 3/3 | Step 14880/14886 | Loss: 0.1195


Teacher | Epoch 3/3 | Step 14886/14886 | Loss: 0.1529


Teacher Epoch 3 Completed | Dev Macro-F1 (thresh=0.5): 0.0043


In [7]:
print("Reloading Best Teacher Checkpoint for Optimal Threshold Search & Submission...")

# 1. Reload Best Teacher Checkpoint Weights
checkpoint_t = torch.load(TEACHER_CHECKPOINT_PATH, map_location=device)
teacher_model.load_state_dict(checkpoint_t["state_dict"], strict=False)
teacher_model.eval()
print(f"Loaded Best Teacher Model from Epoch {checkpoint_t['epoch']} (Dev Macro-F1: {checkpoint_t['best_f1']:.4f})\n")

# 2. Optimal Decision Threshold Search for Teacher on Dev Set
dev_logits_t, dev_targets_t = [], []
with torch.no_grad():
    for batch in dev_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        outputs = teacher_model(input_ids=ids, attention_mask=mask)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        dev_logits_t.append(logits.cpu())
        dev_targets_t.append(batch["labels"])

dev_logits_t = torch.cat(dev_logits_t, dim=0).numpy()
dev_targets_t = torch.cat(dev_targets_t, dim=0).numpy()
probs_t = 1.0 / (1.0 + np.exp(-dev_logits_t))

best_thresh_teacher = 0.5
best_macro_f1_teacher = 0.0

for thresh in np.arange(0.10, 0.90, 0.05):
    preds = (probs_t >= thresh).astype(int)
    score = f1_score(dev_targets_t, preds, average="macro", zero_division=0)
    if score > best_macro_f1_teacher:
        best_macro_f1_teacher = score
        best_thresh_teacher = thresh

print(f"Optimal Teacher Decision Threshold: {best_thresh_teacher:.2f} (Dev Macro-F1: {best_macro_f1_teacher:.4f})")

# 3. Test Set Inference for Teacher Model
print("\nGenerating Test Predictions for Teacher Model...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_5_LSI/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

if DEBUG and CFG["test_samples"]:
    test_data = test_data[:CFG["test_samples"]]

predictions_teacher = []
for case in test_data:
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    inputs = tokenizer(case_text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = teacher_model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        prob_vec = torch.sigmoid(logits.squeeze(0)).cpu().numpy()
        
        pred_indices = np.where(prob_vec >= best_thresh_teacher)[0].tolist()
        predicted_statutes = [int(IDX2STATUTE[idx]) for idx in pred_indices]
        
        if not predicted_statutes:
            top_idx = int(prob_vec.argmax())
            predicted_statutes = [int(IDX2STATUTE[top_idx])]
            
    predictions_teacher.append({
        "id": case_id,
        "pred": sorted(list(set(predicted_statutes)))
    })

# Export Teacher Submission Zip
output_dir_t = "./phase5_teacher_lorma"
os.makedirs(output_dir_t, exist_ok=True)
pred_path_t = os.path.join(output_dir_t, "prediction.json")
sub_zip_path_t = os.path.join(output_dir_t, "submission.zip")

with open(pred_path_t, "w", encoding="utf-8") as f:
    json.dump(predictions_teacher, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path_t, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path_t, arcname="prediction.json")

print(f"Teacher Submission successfully saved to: {sub_zip_path_t}\n")

Reloading Best Teacher Checkpoint for Optimal Threshold Search & Submission...
Loaded Best Teacher Model from Epoch 1 (Dev Macro-F1: 0.1036)



Optimal Teacher Decision Threshold: 0.10 (Dev Macro-F1: 0.0642)

Generating Test Predictions for Teacher Model...


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Teacher Submission successfully saved to: ./phase5_teacher_lorma/submission.zip



In [8]:
# 1. Instantiate Small Student Model (Scaled-down Modern Decoder: 3 layers, 384 hidden_size)
student_model = CustomModernDecoderSequenceClassification(
    vocab_size=len(tokenizer), 
    num_labels=NUM_STATUTES, 
    hidden_size=384, 
    num_layers=3, 
    num_heads=6, 
    intermediate_size=1024
).to(device)

student_total = sum(p.numel() for p in student_model.parameters())
print(f"[Student Model Summary]:")
print(f"  - Total Parameters: {student_total:,} (vs Teacher {teacher_total:,} -> {student_total/teacher_total*100:.1f}% size)")

STUDENT_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_student_kd.pt")
student_optimizer = torch.optim.AdamW(student_model.parameters(), lr=5e-4, weight_decay=0.01)

# Distillation Hyperparameters
TEMPERATURE = 2.0
ALPHA_DISTILL = 0.5

bce_loss_fn = nn.BCEWithLogitsLoss()

def multilabel_kd_loss(student_logits, teacher_logits, hard_labels, temperature=2.0, alpha=0.5):
    hard_loss = bce_loss_fn(student_logits, hard_labels)
    teacher_probs = torch.sigmoid(teacher_logits / temperature)
    soft_loss = F.binary_cross_entropy_with_logits(
        student_logits / temperature, teacher_probs
    ) * (temperature ** 2)
    
    return (1.0 - alpha) * hard_loss + alpha * soft_loss

best_student_f1 = 0.0
print(f"\nCommencing Knowledge Distillation Training ({CFG['student_epochs']} Epochs)...")

for epoch in range(CFG["student_epochs"]):
    student_model.train()
    total_kd_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        student_optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Get Teacher Logits without gradients
        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_logits = teacher_outputs["logits"] if isinstance(teacher_outputs, dict) else teacher_outputs
            
        # Get Student Logits
        student_outputs = student_model(input_ids=input_ids, attention_mask=attention_mask)
        student_logits = student_outputs["logits"] if isinstance(student_outputs, dict) else student_outputs
        
        # Calculate KD Loss
        loss = multilabel_kd_loss(
            student_logits=student_logits, 
            teacher_logits=teacher_logits, 
            hard_labels=labels, 
            temperature=TEMPERATURE, 
            alpha=ALPHA_DISTILL
        )
        
        loss.backward()
        student_optimizer.step()
        total_kd_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Student KD | Epoch {epoch+1}/{CFG['student_epochs']} | Step {step+1}/{len(train_loader)} | KD Loss: {loss.item():.4f}")
            
    # Dev Set Validation for Student Model
    student_model.eval()
    dev_logits, dev_targets = [], []
    with torch.no_grad():
        for batch in dev_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            outputs = student_model(input_ids=ids, attention_mask=mask)
            dev_logits.append(outputs["logits"].cpu() if isinstance(outputs, dict) else outputs.cpu())
            dev_targets.append(batch["labels"])
            
    dev_logits = torch.cat(dev_logits, dim=0).numpy()
    dev_targets = torch.cat(dev_targets, dim=0).numpy()
    probs = 1.0 / (1.0 + np.exp(-dev_logits))
    preds = (probs >= 0.5).astype(int)
    
    epoch_f1 = f1_score(dev_targets, preds, average="macro", zero_division=0)
    print(f"Student Epoch {epoch+1} Completed | Dev Macro-F1 (thresh=0.5): {epoch_f1:.4f}")
    
    if epoch_f1 > best_student_f1 or epoch == 0:
        best_student_f1 = epoch_f1
        torch.save({"epoch": epoch + 1, "best_f1": best_student_f1, "state_dict": student_model.state_dict()}, STUDENT_CHECKPOINT_PATH)
        print(f"  [+] Saved Best Student Checkpoint to '{STUDENT_CHECKPOINT_PATH}'")

[Student Model Summary]:
  - Total Parameters: 17,637,504 (vs Teacher 68,530,944 -> 25.7% size)

Commencing Knowledge Distillation Training (5 Epochs)...


Student KD | Epoch 1/5 | Step 20/14886 | KD Loss: 0.8261


Student KD | Epoch 1/5 | Step 40/14886 | KD Loss: 0.8091


Student KD | Epoch 1/5 | Step 60/14886 | KD Loss: 0.7990


Student KD | Epoch 1/5 | Step 80/14886 | KD Loss: 0.8453


Student KD | Epoch 1/5 | Step 100/14886 | KD Loss: 0.8075


... (2,359 outputs left out)

Student KD | Epoch 4/5 | Step 2540/14886 | KD Loss: 0.8140


Student KD | Epoch 4/5 | Step 2560/14886 | KD Loss: 0.7881


Student KD | Epoch 4/5 | Step 2580/14886 | KD Loss: 0.8134


Student KD | Epoch 4/5 | Step 2600/14886 | KD Loss: 0.8048


In [ ]:
print("Reloading Best Student Checkpoint for Optimal Threshold Search & Submission...")

# 1. Reload Best Student Checkpoint Weights
checkpoint_s = torch.load(STUDENT_CHECKPOINT_PATH, map_location=device)
student_model.load_state_dict(checkpoint_s["state_dict"])
student_model.eval()
print(f"Loaded Best Student Model from Epoch {checkpoint_s['epoch']} (Dev Macro-F1: {checkpoint_s['best_f1']:.4f})\n")

# 2. Optimal Decision Threshold Search for Student on Dev Set
all_logits_s, all_targets_s = [], []
with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        
        outputs = student_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        
        all_logits_s.append(logits.cpu())
        all_targets_s.append(labels)

all_logits_s = torch.cat(all_logits_s, dim=0).numpy()
all_targets_s = torch.cat(all_targets_s, dim=0).numpy()
probs_s = 1.0 / (1.0 + np.exp(-all_logits_s))

best_thresh_student = 0.5
best_macro_f1_student = 0.0

for thresh in np.arange(0.10, 0.90, 0.05):
    preds = (probs_s >= thresh).astype(int)
    score = f1_score(all_targets_s, preds, average="macro", zero_division=0)
    if score > best_macro_f1_student:
        best_macro_f1_student = score
        best_thresh_student = thresh

print(f"Optimal Student Decision Threshold: {best_thresh_student:.2f} (Distilled Student Dev Macro-F1: {best_macro_f1_student:.4f})")

# 3. Test Set Inference for Student Model
print("\nGenerating Test Predictions for Student Model...")
predictions_student = []

for case in test_data:
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    inputs = tokenizer(case_text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = student_model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        prob_vec = torch.sigmoid(logits.squeeze(0)).cpu().numpy()
        
        pred_indices = np.where(prob_vec >= best_thresh_student)[0].tolist()
        predicted_statutes = [int(IDX2STATUTE[idx]) for idx in pred_indices]
        
        if not predicted_statutes:
            top_idx = int(prob_vec.argmax())
            predicted_statutes = [int(IDX2STATUTE[top_idx])]
            
    predictions_student.append({
        "id": case_id,
        "pred": sorted(list(set(predicted_statutes)))
    })

# Export Student Submission Zip
output_dir_s = "./phase5_student_kd"
os.makedirs(output_dir_s, exist_ok=True)
pred_path_s = os.path.join(output_dir_s, "prediction.json")
sub_zip_path_s = os.path.join(output_dir_s, "submission.zip")

with open(pred_path_s, "w", encoding="utf-8") as f:
    json.dump(predictions_student, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path_s, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path_s, arcname="prediction.json")

print(f"Student Submission successfully saved to: {sub_zip_path_s}")

if DEBUG:
    print("\n[DEBUG] Sample Student Prediction Output:")
    print(json.dumps(predictions_student[:2], indent=2))